# Session 3 — Behavior (clicks / keys / RT), GSR, and pupil  
**Duration:** 1.5 hours

### Learning goals
1. Align **gaze** with **mouse** and stimulus events.
2. Compute simple response-time style metrics (stimulus → click).
3. Read Tobii **GSR** / SCR columns and discuss confounds.
4. Analyze **pupil** + **blinks** (Tobii food sample + Pupil Labs export).


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SESSION_DIR = Path.cwd()
WORKSHOP_DIR = SESSION_DIR.parent if SESSION_DIR.name == "sessions" else SESSION_DIR / "workshop"
sys.path.insert(0, str(WORKSHOP_DIR))
from analysis.paths import data_path

## 1. Multimodal timeline on the food task (20 min)

In [ ]:
raw = pd.read_csv(data_path("food_decision_making", "Food_Decision_Making_Teaching_Sample.csv"))
raw["time_s"] = raw["Recording timestamp"].astype(float) / 1e3  # ms → s in this export

events = raw.dropna(subset=["Event"]).copy()
events[["time_s", "Event", "Event value", "Presented Stimulus name"]].head(20)

In [ ]:
# Stimulus intervals from ImageStimulusStart / End
starts = events.query("Event == 'ImageStimulusStart'")[["time_s", "Event value"]].rename(
    columns={"time_s": "start_s", "Event value": "stimulus"}
)
ends = events.query("Event == 'ImageStimulusEnd'")[["time_s", "Event value"]].rename(
    columns={"time_s": "end_s", "Event value": "stimulus"}
)
# Pair by order within stimulus name (teaching approach)
intervals = []
for stim, sgrp in starts.groupby("stimulus"):
    e = ends[ends["stimulus"] == stim].sort_values("end_s")
    s = sgrp.sort_values("start_s")
    for (_, row_s), (_, row_e) in zip(s.iterrows(), e.iterrows()):
        intervals.append({"stimulus": stim, "start_s": row_s["start_s"], "end_s": row_e["end_s"]})
intervals = pd.DataFrame(intervals)
intervals

In [ ]:
mouse = raw.query("Sensor == 'Mouse'").dropna(subset=["Mouse position X", "Mouse position Y"]).copy()
clicks = events.query("Event == 'MouseEvent'").copy()
clicks.head()

## 2. Response times: stimulus onset → first click (20 min)

Not every export has a dedicated RT column — we **construct** RT from the timeline.


In [ ]:
# First mouse event after each food stimulus start (skip instruction/fixation/thanks)
foods = ["cake", "pizza", "ice-cream", "cereal", "date"]
rt_rows = []
for _, iv in intervals.query("stimulus in @foods").iterrows():
    c = clicks[(clicks["time_s"] >= iv["start_s"]) & (clicks["time_s"] <= iv["end_s"])]
    if len(c) == 0:
        continue
    first = c.sort_values("time_s").iloc[0]
    rt_rows.append({
        "stimulus": iv["stimulus"],
        "rt_s": float(first["time_s"] - iv["start_s"]),
        "click_value": first.get("Event value"),
    })
rt = pd.DataFrame(rt_rows)
rt

In [ ]:
if len(rt):
    ax = rt.plot(x="stimulus", y="rt_s", kind="bar", legend=False, color="#8c2d4a")
    ax.set_ylabel("RT (s) — onset to first click")
    ax.set_title("Constructed response times")
    plt.tight_layout()
    plt.show()
else:
    print("No click-aligned RTs in this teaching sample — discuss why.")

## 3. GSR metrics table (20 min)

We use Tobii’s **aggregated** GSR metrics (classroom-friendly). Raw EDA pipelines (neurokit2) can be a bonus demo.


In [ ]:
gsr = pd.read_csv(data_path("tobii_gsr_demo", "Tobii_Pro_Lab_GSR_Demo_Project_Metrics.tsv"), sep="\t")
cols = [
    "Recording", "Participant", "TOI", "Media", "Average_GSR", "Number_of_SCR",
    "Amplitude_of_event_related_SCR", "Average_whole-fixation_pupil_diameter",
    "Last_key_press",
]
cols = [c for c in cols if c in gsr.columns]
g = gsr[cols].copy()
# Coerce numeric fields that may contain strings
for c in ["Average_GSR", "Number_of_SCR", "Amplitude_of_event_related_SCR", "Average_whole-fixation_pupil_diameter"]:
    if c in g.columns:
        g[c] = pd.to_numeric(g[c], errors="coerce")
g.head()

In [ ]:
# Participant-level GSR summary
part = g.groupby("Participant", as_index=False).agg(
    mean_gsr=("Average_GSR", "mean"),
    mean_scr=("Number_of_SCR", "mean"),
    mean_pupil=("Average_whole-fixation_pupil_diameter", "mean"),
)
part.head()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(part["mean_gsr"], part["mean_scr"], alpha=0.8)
ax.set_xlabel("Mean Average_GSR")
ax.set_ylabel("Mean Number_of_SCR")
ax.set_title("GSR level vs SCR count (participants)")
plt.tight_layout()
plt.show()

## 4. Pupil diameter on the food sample (15 min)

In [ ]:
gaze = raw.query("Sensor == 'Eye Tracker'").copy()
gaze["time_s"] = gaze["Recording timestamp"].astype(float) / 1e3
pup = gaze.dropna(subset=["Pupil diameter left"]).copy()
pup["pupil"] = pup[["Pupil diameter left", "Pupil diameter right"]].mean(axis=1, skipna=True)
# Simple baseline: first 0.5 s of each stimulus (if available)
rows = []
for stim, gstim in pup.groupby("Presented Stimulus name"):
    gstim = gstim.sort_values("time_s")
    t0 = gstim["time_s"].iloc[0]
    base = gstim.loc[gstim["time_s"] <= t0 + 0.5, "pupil"].mean()
    gstim = gstim.copy()
    gstim["pupil_baseline_corrected"] = gstim["pupil"] - base
    rows.append(gstim)
pup_c = pd.concat(rows, ignore_index=True)
pup_c.groupby("Presented Stimulus name")["pupil_baseline_corrected"].mean().sort_values()

In [ ]:
# Blink awareness from Pupil Labs export
blinks = pd.read_csv(data_path("pupil_labs_recording", "blinks.csv"))
blinks["duration"].describe()

## 5. Combining gaze + behavior (15 min)

Classic teaching metric already present in GSR metrics:  
`Time_from_first_fixation_to_mouse_click.*`

We also build a transparent version on the food sample when AOI hit columns exist.


In [ ]:
aoi_cols = [c for c in raw.columns if c.startswith("AOI hit")]
print("AOI hit columns:", len(aoi_cols))
aoi_cols[:8]

## Practice
1. Plot pupil (baseline-corrected) over time for `cake`.
2. List three confounds for GSR in a lab with talking + movement.
3. Explain why RT from mouse events can disagree with Tobii “time to first click” metrics.

### Exit ticket
One multimodal question you could publish as a figure caption.
